# VAJRA — Qwen3-Coder 30B A3B T4×2 Worker

Fresh worker appliance. Qwen3-Coder-30B-A3B-Instruct Q4_K_M GGUF + llama.cpp CUDA + the existing `vajra-worker-v1` protocol. The model is proposal-only; VAJRA remains authoritative for policy, execution, verification, and Run state.

**Runtime:** Kaggle 2× Tesla T4, Internet ON. The cell stays alive while the Kaggle runtime is alive; Kaggle can still reclaim/destroy the runtime.


In [ ]:
%%bash
set -euo pipefail

REPO_DIR=/kaggle/working/vajra
MODEL_REPO=tensorblock/Qwen_Qwen3-Coder-30B-A3B-Instruct-GGUF
MODEL_FILE_NAME=Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf
MODEL_FILE=/kaggle/working/models/$MODEL_FILE_NAME
LLAMA_DIR=/kaggle/working/llama
LLAMA_API=http://127.0.0.1:8000
WORKER_API=http://127.0.0.1:8787
MODEL_ID=Qwen3-Coder-30B-A3B-Instruct-Q4_K_M

echo '=== VAJRA QWEN3-CODER T4x2 WORKER ==='
nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
GPU_COUNT=$(nvidia-smi --query-gpu=name --format=csv,noheader | grep -c 'Tesla T4' || true)
test "$GPU_COUNT" -ge 2

echo '=== VAJRA SOURCE ==='
rm -rf "$REPO_DIR"
mkdir -p /kaggle/working
curl -fsSL https://github.com/Exploiter69/vajra/archive/refs/heads/main.tar.gz | tar -xz -C /kaggle/working
mv /kaggle/working/vajra-main "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src"
cd "$REPO_DIR"
git rev-parse --short HEAD || true

echo '=== MODEL ==='
mkdir -p /kaggle/working/models
if [ ! -s "$MODEL_FILE" ]; then
  python -m pip install -q -U huggingface_hub
  HF_HUB_ENABLE_HF_TRANSFER=1 python - <<'PY'
from huggingface_hub import hf_hub_download
path = hf_hub_download(
    repo_id='tensorblock/Qwen_Qwen3-Coder-30B-A3B-Instruct-GGUF',
    filename='Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf',
    local_dir='/kaggle/working/models',
)
print(path)
PY
fi
test -s "$MODEL_FILE"
du -h "$MODEL_FILE"

echo '=== LLAMA.CPP CUDA ==='
mkdir -p "$LLAMA_DIR"
if [ ! -x "$LLAMA_DIR/llama-server" ]; then
  python - <<'PY'
import json, urllib.request
release='b10982'
data=json.load(urllib.request.urlopen(f'https://api.github.com/repos/ggml-org/llama.cpp/releases/tags/{release}'))
assets=data['assets']
matches=[a for a in assets if 'ubuntu-cuda-13-x64' in a['name'] and a['name'].endswith(('.tar.gz','.tgz','.zip'))]
if not matches:
    raise SystemExit('No Ubuntu CUDA 13 x64 llama.cpp release asset found')
a=matches[0]
print(a['browser_download_url'])
open('/kaggle/working/llama_asset_url','w').write(a['browser_download_url'])
PY
  LLAMA_ASSET_URL=$(cat /kaggle/working/llama_asset_url)
  curl -fL "$LLAMA_ASSET_URL" -o /kaggle/working/llama.tar.gz
  case "$LLAMA_ASSET_URL" in
    *.tar.gz) tar -xzf /kaggle/working/llama.tar.gz -C "$LLAMA_DIR" --strip-components=1 ;;
    *.tgz) tar -xzf /kaggle/working/llama.tar.gz -C "$LLAMA_DIR" --strip-components=1 ;;
    *) echo 'Unsupported llama.cpp asset format'; exit 1 ;;
  esac
fi
LLAMA_SERVER=$(find "$LLAMA_DIR" -type f -name llama-server -perm -111 | head -n1)
test -n "$LLAMA_SERVER"
echo "llama-server=$LLAMA_SERVER"
"$LLAMA_SERVER" --version || true
"$LLAMA_SERVER" --list-devices

echo '=== START LLAMA.CPP ==='
"$LLAMA_SERVER" \
  -m "$MODEL_FILE" \
  --host 127.0.0.1 --port 8000 \
  --n-gpu-layers all \
  --split-mode layer \
  --tensor-split 1,1 \
  --ctx-size 8192 \
  --parallel 1 \
  --metrics \
  > /kaggle/working/llama-server.log 2>&1 &
LLAMA_PID=$!
trap 'kill "$LLAMA_PID" 2>/dev/null || true; kill "$WORKER_PID" 2>/dev/null || true' EXIT INT TERM

for i in $(seq 1 180); do
  if curl -fsS "$LLAMA_API/health" >/dev/null 2>&1; then break; fi
  if ! kill -0 "$LLAMA_PID" 2>/dev/null; then cat /kaggle/working/llama-server.log; exit 1; fi
  sleep 2
done
curl -fsS "$LLAMA_API/health"

echo '=== START VAJRA WORKER ==='
export VAJRA_WORKER_HOST=127.0.0.1
export VAJRA_WORKER_PORT=8787
export VAJRA_LLAMA_URL="$LLAMA_API"
export VAJRA_WORKER_MODEL="$MODEL_ID"
export VAJRA_WORKER_ID=kaggle-qwen3-coder-t4x2-01
python -m vajra.runtime.llama_cpp_worker_server >/kaggle/working/vajra-worker.log 2>&1 &
WORKER_PID=$!

for i in $(seq 1 60); do
  if curl -fsS "$WORKER_API/health" >/dev/null 2>&1; then break; fi
  if ! kill -0 "$WORKER_PID" 2>/dev/null; then cat /kaggle/working/vajra-worker.log; exit 1; fi
  sleep 1
done
curl -fsS "$WORKER_API/health"
curl -fsS "$WORKER_API/capabilities"

echo '=== REAL VAJRA PROTOCOL SMOKE ==='
python - <<'PY'
import json, os, time, urllib.request
from vajra.runtime.kaggle_worker import KaggleWorkerAdapter
from vajra.runtime.worker_protocol import WorkerJob
   
job=WorkerJob(
    run_id='kaggle-qwen3-smoke', step_id='smoke-1', attempt_id='attempt-1',
    repository_revision='not-applicable',
    workspace_contract={'mode':'proposal-only','authority':'vajra-control-plane'},
    context_bundle={'prompt':'Return exactly the text VAJRA_QWEN3_CODER_OK'},
    allowed_capabilities=(),
    budget={'max_output_tokens':64,'timeout_seconds':180},
    deadline='2030-01-01T00:00:00Z', expected_output_schema={'type':'string'},
   correlation_id='kaggle-qwen3-correlation-001')
adapter=KaggleWorkerAdapter()
req=urllib.request.Request('http://127.0.0.1:8787/infer',data=adapter.encode_job(job).encode(),headers={'Content-Type':'application/json'},method='POST')
started=time.time()
with urllib.request.urlopen(req,timeout=190) as r: payload=json.loads(r.read().decode())
print(json.dumps(payload,indent=2,sort_keys=True))
assert payload['status']=='completed'
assert payload['correlation_id']=='kaggle-qwen3-correlation-001'
assert 'VAJRA_QWEN3_CODER_OK' in payload['structured_result']['response']
print(f'elapsed_seconds={time.time()-started:.3f}')
PY

echo '=== VAJRA QWEN3-CODER WORKER READY ==='
echo 'worker=http://127.0.0.1:8787'
echo 'llama=http://127.0.0.1:8000'
echo 'model=Qwen3-Coder-30B-A3B-Instruct-Q4_K_M'
echo 'Keep this cell running while VAJRA uses the worker.'
wait "$LLAMA_PID"
